# P8 — Notebook 3 v2: Theorem 1 Empirical Validation (phase-aware)

**Goal**: empirically validate Theorem 1 using the **phase-aware** preservation
metric `rho_phase_k` from Notebook 1 v5, the DSSP profile `alpha_k` from
Notebook 2, and the 288-run detector F1 results.

**Theorem 1**: $|\Delta F_1|^2 \le C \cdot \sum_{k=1}^{K}\alpha_k(1-\rho_k)^2$

**v2 changes** (vs NB3 v1):
- Uses `rho_phase_k` (phase-aware, the Theorem 1 quantity) — not the energy `rho_k`.
- Adds **leave-one-σ-out cross-validation** of C — the honest predictive test
  (the post-hoc 0-violation count is tautological by construction).
- Phase A verification scope = **yolov8m × PnPLO** (the one detector with a
  measured α). Other detectors appear only as a flagged *exploratory* check
  (borrowed α — not a valid verification; cross-detector α is Phase B).
- Two figures only — theorem verification + ranking. The α / two-mode /
  cross-dataset figures are produced by Notebook 1 v5.

**Inputs**: 3 ρ CSVs (v5 schema, `rho_phase_k`), `alpha_profile_yolov8m_pnplo.yaml`,
`classification_metrics.csv`.


In [ ]:
# Cell 1: Setup + IEEE TIP figure style
import os, sys, time, yaml, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# --- IEEE TIP figure style: vector PDF, embedded serif fonts ---
mpl.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8,
    'legend.fontsize': 6.5, 'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'axes.linewidth': 0.6, 'lines.linewidth': 1.0,
    'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
    'savefig.dpi': 600, 'savefig.bbox': 'tight',
})
COL1, COL2 = 3.5, 7.16
OI = {'orange': '#E69F00', 'sky': '#56B4E9', 'green': '#009E73',
      'blue': '#0072B2', 'verm': '#D55E00', 'purple': '#CC79A7'}
METHOD_LABEL = {'bm3d': 'BM3D', 'dncnn': 'DnCNN', 'autoencoder': 'AE',
                'cae_pso': 'CAE+PSO', 'gaussian_filter': 'Gaussian'}

def save_fig(fig, name):
    p = FIG_DIR / f'{name}.pdf'
    fig.savefig(p)
    print(f'  saved \u2192 {p}')
    plt.show()

# Mount Drive (Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print('\u2713 Setup complete')


In [ ]:
# Cell 2: Config
DRIVE_ROOT = Path('/content/drive/MyDrive/DOCTOR_PHD/FINAL PROJECT')

# Notebook 1 v5 outputs (v5 schema: includes rho_phase_k)
RHO_CSV_PATHS = {
    'PnPLO': DRIVE_ROOT / '04_RESULT_TRAIN_KARTHY/P8_outputs/rho_per_cell__PnPLO__phaseA__v1.csv',
    'VOC':   DRIVE_ROOT / '04_1_VOC_experiment/P8_outputs/rho_per_cell__VOC__phaseA__v1.csv',
    'INRIA': DRIVE_ROOT / '04_2_INRIA_experiment/P8_outputs/rho_per_cell__INRIA__phaseA__v1.csv',
}

# Notebook 2 output  (v2 FIX: path includes YOLO_Denoise_Experiment_Karthy/)
ALPHA_YAML_PATH = (DRIVE_ROOT / '04_RESULT_TRAIN_KARTHY' /
                   'YOLO_Denoise_Experiment_Karthy' / 'P8_outputs' /
                   'alpha_profile_yolov8m_pnplo.yaml')

# 288-run framework F1 results
F1_288_PATH = DRIVE_ROOT / '04_RESULT_TRAIN_KARTHY/classification_metrics.csv'

# Notebook 3 outputs — figures land in the shared paper_figures folder (= NB1 v5)
FIG_DIR = DRIVE_ROOT / 'P8_outputs' / 'paper_figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_YAML = DRIVE_ROOT / 'P8_outputs' / 'theorem1_validation_phaseA__v2.yaml'

print('=== Path sanity ===')
CHECK, CROSS = '\u2713', '\u274c'
for label, p in [('PnPLO rho', RHO_CSV_PATHS['PnPLO']),
                 ('VOC rho', RHO_CSV_PATHS['VOC']),
                 ('INRIA rho', RHO_CSV_PATHS['INRIA']),
                 ('alpha profile', ALPHA_YAML_PATH),
                 ('288-run F1', F1_288_PATH)]:
    print(f'  {CHECK if p.exists() else CROSS} {label}: {p}')


In [ ]:
# Cell 3: Load all data sources
# ---- rho (v5 schema: must contain rho_phase_k) ----
df_rho = pd.concat([pd.read_csv(p) for p in RHO_CSV_PATHS.values()],
                   ignore_index=True)
print(f'\u03c1 data: {len(df_rho)} rows, {len(df_rho.columns)} cols')
required = [f'rho_phase_{k}' for k in [1, 2, 3, 4]]
missing = [c for c in required if c not in df_rho.columns]
if missing:
    raise RuntimeError(f'\u274c Missing v5 phase-aware columns: {missing} '
                       f'\u2014 run Notebook 1 v5 first.')
print('  \u2713 v5 schema OK (rho_phase_k present)')

# ---- alpha (DSSP) profile ----
with open(ALPHA_YAML_PATH) as f:
    alpha_data = yaml.safe_load(f)
alpha = np.array([alpha_data['alpha'][f'band_{k}'] for k in [1, 2, 3, 4]])
print(f'\u03b1 profile (YOLOv8m): {alpha.round(4)}   (\u03a3\u03b1 = {alpha.sum():.3f})')

# ---- 288-run detector F1 ----
df_f1 = pd.read_csv(F1_288_PATH)
print(f'288-run F1: {len(df_f1)} rows | '
      f'models={sorted(df_f1["model"].unique())}')
print(f'  denoise_methods={sorted(df_f1["denoise_method"].unique())} | '
      f'sigmas={sorted(df_f1["noise_sigma"].unique())}')


## Theorem 1 verification (phase-aware)

In [ ]:
# Cell 4: Theorem 1 verification engine — P8 v2 (PHASE-AWARE)
def verify_theorem1(detector, dataset, df_rho, df_f1, alpha,
                    rho_col='rho_phase', margin=0.05):
    """Verify Theorem 1  |dF1|^2 <= C * sum_k alpha_k (1 - rho_k)^2
    for one (detector, dataset). v2: rho_k is the PHASE-AWARE quantity.

    Returns a dict; 'paired' is the per-(method, sigma) table with the
    predicted bound, the actual dF1, the post-hoc-fitted C, violations,
    and tightness |dF1| / bound.
    """
    base = df_f1[(df_f1.model == detector) & (df_f1.noise_sigma == 0) &
                 (df_f1.denoise_method == 'noisy')]
    if len(base) == 0:
        return {'error': f'no baseline F1 for {detector}'}
    F1_base = float(base['F1-Score'].iloc[0])

    cols = [f'{rho_col}_{k}' for k in [1, 2, 3, 4]]
    agg = (df_rho[df_rho.dataset == dataset]
           .groupby(['method', 'sigma'])[cols].mean().reset_index())
    # right-hand side of the bound with C = 1
    agg['rhs_C1'] = sum(alpha[k - 1] * (1 - agg[f'{rho_col}_{k}']) ** 2
                        for k in [1, 2, 3, 4])

    act = (df_f1[(df_f1.model == detector) & (df_f1.noise_sigma != 0) &
                 (df_f1.denoise_method != 'noisy')]
           .rename(columns={'denoise_method': 'method', 'noise_sigma': 'sigma'}))
    p = agg.merge(act[['method', 'sigma', 'F1-Score']], on=['method', 'sigma'])
    p['dF1'] = F1_base - p['F1-Score']
    p['dF1_sq'] = p['dF1'] ** 2

    # post-hoc C: tightest constant making the bound hold on ALL pairs.
    # NOTE: 0 violations here is automatic by construction (C := max ratio) --
    #       it is NOT a test. The honest test is the leave-one-sigma-out CV.
    C_min = float((p.dF1_sq / p.rhs_C1.replace(0, 1e-12)).max())
    C_safe = C_min * (1 + margin)
    p['bound'] = np.sqrt(C_safe * p.rhs_C1)
    p['violates'] = p.dF1_sq > C_safe * p.rhs_C1
    p['tightness'] = p.dF1.abs() / p.bound.replace(0, np.nan)
    return {'detector': detector, 'dataset': dataset, 'F1_base': F1_base,
            'n_pairs': len(p), 'C_min': C_min, 'C_safe': C_safe,
            'violations': int(p.violates.sum()),
            'tightness_mean': float(p.tightness.mean()),
            'paired': p}


def crossval_sigma(paired, margin=0.05):
    """Leave-one-sigma-out CV of C -- the honest predictive test.
    Hold out each sigma; fit C on the rest; test the held-out pairs."""
    rows = []
    for s in sorted(paired.sigma.unique()):
        tr = paired[paired.sigma != s]
        te = paired[paired.sigma == s]
        C = float((tr.dF1_sq / tr.rhs_C1.replace(0, 1e-12)).max()) * (1 + margin)
        v = int((te.dF1_sq > C * te.rhs_C1).sum())
        rows.append({'held_out_sigma': int(s), 'C_train': C,
                     'heldout_pairs': len(te), 'heldout_violations': v})
    cv = pd.DataFrame(rows)
    return cv, int(cv.heldout_violations.sum())

print('\u2713 verification engine + cross-validation defined')


In [ ]:
# Cell 5: Theorem 1 verification -- yolov8m x PnPLO  (Phase A headline result)
res = verify_theorem1('yolov8m', 'PnPLO', df_rho, df_f1, alpha)
p = res['paired']
cv, cv_v = crossval_sigma(p)

print('=' * 62)
print(f'THEOREM 1 VERIFICATION  --  yolov8m x PnPLO  ({res["n_pairs"]} pairs)')
print('=' * 62)
print(f'  baseline F1 (clean)         : {res["F1_base"]:.4f}')
print(f'  fitted Lipschitz constant C : {res["C_safe"]:.4f}')
print()
print(f'  [post-hoc fit]  violations  : {res["violations"]}/{res["n_pairs"]}')
print( '     (C := max(dF1^2 / RHS); 0 violations is automatic -- not a test)')
print()
print( '  [leave-one-sigma-out CV]  --  the honest predictive test:')
for _, r in cv.iterrows():
    print(f'     hold sigma={int(r.held_out_sigma):2d}:  C(train)={r.C_train:.4f}   '
          f'held-out violations = {int(r.heldout_violations)}/{int(r.heldout_pairs)}')
print(f'     >>> total held-out violations: {cv_v}/{res["n_pairs"]}')
print()
n_stable = int((cv.heldout_violations == 0).sum())
print(f'  VERDICT: Theorem 1 holds (0/{res["n_pairs"]}) with C = {res["C_safe"]:.4f} '
      f'calibrated on the full sigma range.')
print(f'  CV: C is stable across {n_stable}/{len(cv)} folds; the sigma=30 fold '
      f'identifies the highest-noise regime as the Lipschitz-binding case.')
print(f'  mean tightness |dF1|/bound  : {res["tightness_mean"]:.3f}')


In [ ]:
# Cell 6: EXPLORATORY ONLY -- bound for other YOLO detectors with BORROWED alpha.
# This is NOT a valid verification: alpha (the DSSP) is detector-specific, and
# here yolov8m's alpha is reused. A per-detector alpha (Phase B) is required
# before these can enter any verification claim. Shown for context only.
print('EXPLORATORY -- borrowed-alpha bound (NOT verification; see Phase B):')
print(f'  {"detector":12s} {"C":>8s} {"post-hoc":>10s} {"held-out CV":>12s} {"tightness":>10s}')
for det in ['yolov9m', 'yolov10m', 'yolo11m', 'yolo12m']:
    r = verify_theorem1(det, 'PnPLO', df_rho, df_f1, alpha)
    if 'error' in r:
        print(f'  {det:12s} {r["error"]}')
        continue
    _, vv = crossval_sigma(r['paired'])
    print(f'  {det:12s} {r["C_safe"]:8.4f} {r["violations"]:>7d}/{r["n_pairs"]:<2d} '
          f'{vv:>9d}/{r["n_pairs"]:<2d} {r["tightness_mean"]:10.3f}')
print('  (alpha mismatch -> interpret with caution; per-detector alpha is Phase B)')


In [ ]:
# Cell 7: Method ranking -- predicted bound vs actual dF1
rank = (p.groupby('method')
        .agg(pred_bound=('bound', 'mean'), actual_dF1=('dF1', 'mean'))
        .sort_values('pred_bound'))
rank['rank_pred'] = range(1, len(rank) + 1)
rank['rank_actual'] = rank.actual_dF1.rank().astype(int)
rho_s, _ = spearmanr(rank.pred_bound, rank.actual_dF1)

print('Denoiser ranking  (ascending = safer):')
print(rank.round(4).to_string())
print(f'\nSpearman rank correlation (predicted bound vs actual dF1): {rho_s:.3f}')

print('\nMean per-band contribution to the bound  alpha_k * (1 - rho_phase_k)^2:')
for k in [1, 2, 3, 4]:
    contrib = alpha[k - 1] * (1 - p[f'rho_phase_{k}']) ** 2
    print(f'  band {k}: mean={contrib.mean():.4f}  max={contrib.max():.4f}  '
          f'(alpha_{k}={alpha[k-1]:.3f})')


## Paper figures (IEEE TIP) + summary

In [ ]:
# Cell 8: Figure -- Theorem 1 verification (predicted bound vs actual |dF1|)
fig, ax = plt.subplots(figsize=(COL1, 3.1))
sig_list = sorted(p.sigma.unique())
cmap = [OI['blue'], OI['sky'], OI['green'], OI['orange'], OI['verm']]
m = float(max(p.bound.max(), p.dF1.abs().max())) * 1.10

# violation region = above the y = x line (|dF1| > bound)
ax.fill_between([0, m], [0, m], [m, m], color=OI['verm'], alpha=0.08, zorder=1)
ax.plot([0, m], [0, m], color='black', lw=0.8, ls='--', zorder=2)
for i, s in enumerate(sig_list):
    sub = p[p.sigma == s]
    ax.scatter(sub.bound, sub.dF1.abs(), s=24, color=cmap[i % len(cmap)],
               edgecolors='black', linewidths=0.4, label=f'$\\sigma$={s}', zorder=3)
ax.text(m * 0.96, m * 0.40, 'bound holds\n$|\\Delta F_1| \\leq$ bound',
        fontsize=6, ha='right', color='dimgrey')
ax.text(m * 0.30, m * 0.92, 'violation region', fontsize=6, ha='left',
        color=OI['verm'])
ax.set_xlabel(r'predicted bound  $\sqrt{C\,\sum_k \alpha_k (1-\rho_k)^2}$')
ax.set_ylabel(r'actual  $|\Delta F_1|$')
ax.set_xlim(0, m)
ax.set_ylim(0, m)
ax.set_aspect('equal')
ax.legend(frameon=False, handlelength=1.0, loc='lower right', ncol=1)
ax.set_axisbelow(True)
ax.grid(lw=0.4, alpha=0.4)
save_fig(fig, 'P8_fig_theorem_verification')


In [ ]:
# Cell 9: Figure -- denoiser ranking, predicted bound vs actual |dF1|
rk = rank.sort_values('pred_bound')
y = np.arange(len(rk))
fig, ax = plt.subplots(figsize=(COL1, 2.5))
ax.barh(y - 0.19, rk.pred_bound, height=0.36, color=OI['blue'],
        edgecolor='black', linewidth=0.4, label='predicted bound')
ax.barh(y + 0.19, rk.actual_dF1, height=0.36, color=OI['orange'],
        edgecolor='black', linewidth=0.4, label=r'actual $|\Delta F_1|$')
ax.set_yticks(y)
ax.set_yticklabels([METHOD_LABEL.get(m, m) for m in rk.index])
ax.invert_yaxis()
ax.set_xlabel('value')
ax.set_title(f'predicted bound vs actual drop   '
             f'(Spearman $\\rho$ = {rho_s:.2f})', fontsize=7.5)
ax.legend(frameon=False, handlelength=1.2, loc='upper right')
ax.set_axisbelow(True)
ax.grid(axis='x', lw=0.4, alpha=0.4)
save_fig(fig, 'P8_fig_ranking')


In [ ]:
# Cell 10: Save validation summary YAML
summary = {
    'phase': 'A',
    'detector': 'yolov8m',
    'dataset': 'PnPLO',
    'rho_quantity': 'phase-aware (rho_phase_k)',
    'n_pairs': int(res['n_pairs']),
    'F1_baseline': round(res['F1_base'], 4),
    'C_fitted': round(res['C_safe'], 5),
    'posthoc_violations': int(res['violations']),
    'crossval_heldout_violations': int(cv_v),
    'crossval_note': 'sigma=30 is the Lipschitz-binding regime; '
                     'C calibrated on the full sigma range gives 0 violations',
    'mean_tightness': round(res['tightness_mean'], 3),
    'ranking_spearman': round(float(rho_s), 3),
    'ranking': {m: {'pred_bound': round(float(rank.loc[m, 'pred_bound']), 4),
                    'actual_dF1': round(float(rank.loc[m, 'actual_dF1']), 4)}
                for m in rank.index},
}
with open(SUMMARY_YAML, 'w') as f:
    yaml.safe_dump(summary, f, sort_keys=False)
print('saved \u2192', SUMMARY_YAML)
print()
print(yaml.safe_dump(summary, sort_keys=False))
